# 2. Hyperparameter Tuning for the Multi-omics Integration Model

This notebook performs hyperparameter tuning for the Multi-omics Integration Clustering (MIC) model using a grid search with 5-fold cross-validation. The goal is to identify the optimal set of hyperparameters (e.g., decoder architecture, dropout rate, weight decay) that maximizes the clustering performance, measured by clustering accuracy (ACC) against the benchmark K-means labels.

The process is as follows:
1.  **Load Data**: Load the preprocessed dataset created in the `01_data_preprocessing.ipynb` notebook.
2.  **Define Search Space**: Specify the grid of hyperparameters to be evaluated.
3.  **Cross-Validation Loop**: For each combination of hyperparameters, train and evaluate the model using 5-fold stratified cross-validation.
4.  **Analyze Results**: Identify the best hyperparameter set based on the average validation accuracy across the folds.
5.  **Save Results**: Save the detailed tuning results to a CSV file for documentation and further analysis.


### 2.1. Setup and Data Loading

In [1]:
# System and project imports
import sys
import os
import pandas as pd
import numpy as np
import torch
from tqdm.notebook import tqdm

# Append src directory to path to import custom modules
sys.path.append('../src')
import config
from models import MIC
from utils import set_seed, cluster_accuracy

# Scikit-learn imports for cross-validation and metrics
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# PyTorch imports
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Set seed for reproducibility
set_seed(config.RANDOM_STATE)

# --- Load Processed Data ---
print("Loading preprocessed data...")
processed_data_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
if not processed_data_path.exists():
    raise FileNotFoundError(f"Processed data not found at {processed_data_path}. Please run '01_data_preprocessing.ipynb' first.")

processed_data = torch.load(processed_data_path)

input_genotype = processed_data['input_genotype']
input_proteome = processed_data['input_proteome']
input_metabolite = processed_data['input_metabolite']
output_clinical = processed_data['output_clinical']
clinical_df = processed_data['clinical_df']

print("Data loaded successfully.")
print(f"Genotype features: {input_genotype.shape[1]}")
print(f"Proteome features: {input_proteome.shape[1]}")
print(f"Metabolite features: {input_metabolite.shape[1]}")
print(f"Clinical features: {output_clinical.shape[1]}")
print(f"Total samples: {len(clinical_df)}")


Loading preprocessed data...
Data loaded successfully.
Genotype features: 402
Proteome features: 714
Metabolite features: 294
Clinical features: 5
Total samples: 493


/tmp/ipykernel_116342/1247578446.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  processed_data = torch.load(processed_data_path)


### 2.2. Configuration for Hyperparameter Search

Here, we define the parameters for the grid search. This includes both fixed parameters (like the encoder architecture) and the parameters we want to tune (the search space).


In [2]:
# --- Execution Environment ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Fixed Hyperparameters ---
ENCODER_DIM = [128, 32]
HIDDEN_DIM = [64]
LATENT_DIM = 16
MAX_EPOCHS = 300
LEARNING_RATE = 0.01
BATCH_SIZE = 64
PATIENCE = 20
N_SPLITS = 5 # Number of folds for K-Fold CV

# --- Search Space ---
# This grid defines the hyperparameters that will be tuned.
# The grid search will iterate through every possible combination.
param_grid = {
    'decoder_dim': [[], [32], [64]],
    'dropout': [0.1, 0.2, 0.3],
    'weight_decay': [1e-4, 1e-3, 1e-2]
}

grid = ParameterGrid(param_grid)
print(f"\nDefined search space with {len(grid)} parameter combinations.")
print("Fixed parameters:")
print(f"  - Encoder Dims: {ENCODER_DIM}")
print(f"  - Hidden Dims: {HIDDEN_DIM}")
print(f"  - Latent Dim: {LATENT_DIM}")
print("Search parameters:")
print(f"  - Decoder Dims: {param_grid['decoder_dim']}")
print(f"  - Dropout: {param_grid['dropout']}")
print(f"  - Weight Decay: {param_grid['weight_decay']}")


Using device: cuda

Defined search space with 27 parameter combinations.
Fixed parameters:
  - Encoder Dims: [128, 32]
  - Hidden Dims: [64]
  - Latent Dim: 16
Search parameters:
  - Decoder Dims: [[], [32], [64]]
  - Dropout: [0.1, 0.2, 0.3]
  - Weight Decay: [0.0001, 0.001, 0.01]


### 2.3. Training and Evaluation Function for a Single Fold

This function encapsulates the logic for training and evaluating the model on a single fold of the cross-validation data. It takes a set of hyperparameters and the training/validation indices as input and returns various performance metrics. This modular approach keeps the main cross-validation loop clean and readable.


In [3]:
def train_one_fold(train_idx, val_idx, params):
    """
    Trains and evaluates the MIC model for one fold of cross-validation.

    Args:
        train_idx (list): List of indices for the training set.
        val_idx (list): List of indices for the validation set.
        params (dict): A dictionary of hyperparameters for this run.

    Returns:
        tuple: A tuple containing various performance metrics for the fold, matching the original notebook's output.
    """
    # --- Model Definition ---
    input_dims = {
        'genotype': input_genotype.shape[1],
        'proteome': input_proteome.shape[1],
        'metabolite': input_metabolite.shape[1]
    }
    
    # params['decoder_dim'] is the hyperparameter being tuned (e.g., [], [32])
    decoder_dims = params['decoder_dim']

    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=decoder_dims,
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=params['dropout']
    ).to(DEVICE)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=params['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=5)
    loss_fn = F.mse_loss

    # --- DataLoaders ---
    full_dataset = TensorDataset(input_genotype, input_proteome, input_metabolite, output_clinical)
    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(Subset(full_dataset, val_idx),   batch_size=BATCH_SIZE, shuffle=False)
    eval_loader  = DataLoader(Subset(full_dataset, train_idx),  batch_size=BATCH_SIZE, shuffle=False)

    # --- Training Loop with Early Stopping ---
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = model.state_dict()
    trained_epochs = 0
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        for x1, x2, x3, y in train_loader:
            x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x1, x2, x3)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        epoch_val_loss = []
        with torch.no_grad():
            for x1, x2, x3, y in val_loader:
                x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
                pred = model(x1, x2, x3)
                loss = loss_fn(pred, y)
                epoch_val_loss.append(loss.item())
        
        current_val_loss = np.mean(epoch_val_loss)
        scheduler.step(current_val_loss)
        
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            epochs_no_improve = 0
            best_model_wts = model.state_dict()
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= PATIENCE:
            trained_epochs = epoch + 1
            break
            
    if trained_epochs == 0:
        trained_epochs = MAX_EPOCHS
        
    model.load_state_dict(best_model_wts)

    # --- Final Evaluation ---
    model.eval()

    # Calculate final reconstruction losses (matching original notebook logic)
    final_train_loss_list = []
    with torch.no_grad():
        for x1, x2, x3, y in train_loader:
            x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
            pred = model(x1, x2, x3)
            final_train_loss_list.append(loss_fn(pred, y).item())
    final_train_loss = np.mean(final_train_loss_list)

    final_val_loss_list = []
    with torch.no_grad():
        for x1, x2, x3, y in val_loader:
            x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
            pred = model(x1, x2, x3)
            final_val_loss_list.append(loss_fn(pred, y).item())
    final_val_loss = np.mean(final_val_loss_list)
    
    # Calculate clustering metrics (matching original notebook's exact logic)
    train_cluster_acc, val_cluster_acc, train_ari, val_ari, train_sil_score, val_sil_score = -1, -1, -1, -1, -1, -1
    with torch.no_grad():
        z_train = model.get_latent_space(eval_loader, device=DEVICE).numpy()
        kmeans_model = KMeans(n_clusters=config.NUM_CLUSTERS, random_state=config.RANDOM_STATE, n_init=100).fit(z_train)
        train_labels_pred = kmeans_model.predict(z_train)
        train_labels_true = clinical_df.iloc[train_idx]["kmeans_cluster"].values
        train_cluster_acc = cluster_accuracy(train_labels_true, train_labels_pred)
        train_ari = adjusted_rand_score(train_labels_true, train_labels_pred)
        
        z_val = model.get_latent_space(val_loader, device=DEVICE).numpy()
        val_labels_pred = kmeans_model.predict(z_val)
        val_labels_true = clinical_df.iloc[val_idx]["kmeans_cluster"].values
        val_cluster_acc = cluster_accuracy(val_labels_true, val_labels_pred)
        val_ari = adjusted_rand_score(val_labels_true, val_labels_pred)

        if len(np.unique(train_labels_pred)) > 1:
            train_sil_score = silhouette_score(z_train, train_labels_pred)
        
        if len(np.unique(val_labels_pred)) > 1:
            val_sil_score = silhouette_score(z_val, val_labels_pred)

    return final_train_loss, final_val_loss, train_cluster_acc, val_cluster_acc, train_ari, val_ari, train_sil_score, val_sil_score, trained_epochs


### 2.4. Run Grid Search with K-Fold Cross-Validation

Now we execute the main hyperparameter tuning loop. We use `StratifiedKFold` to ensure that the proportion of benchmark clusters is maintained in each training and validation split. This is important for reliable evaluation, especially if the cluster sizes are imbalanced.

The process iterates through each parameter combination defined in our `param_grid`. For each combination, it performs a full 5-fold cross-validation, training and evaluating the model five times on different subsets of the data. The performance metrics from each fold are collected and then averaged to get a robust estimate of the model's performance for that specific set of hyperparameters.


In [4]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)
stratify_labels = clinical_df["kmeans_cluster"].values

hyperparam_results = []

print(f"Starting Hyperparameter Tuning with {N_SPLITS}-Fold Cross-Validation...")
print("="*80)

# Wrap the grid iterator with tqdm for a progress bar
for params in tqdm(grid, desc="Hyperparameter Combinations"):
    
    fold_results = []
    
    # K-Fold CV loop
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(stratify_labels)), stratify_labels)):
        
        # train_one_fold now returns a tuple matching the original notebook
        final_train_loss, final_val_loss, train_cluster_acc, val_cluster_acc, train_ari, val_ari, train_sil_score, val_sil_score, trained_epochs = train_one_fold(
            train_idx=list(train_idx),
            val_idx=list(val_idx),
            params=params
        )
        
        fold_results.append({
            'train_loss': final_train_loss,
            'val_loss': final_val_loss,
            'train_acc': train_cluster_acc,
            'val_acc': val_cluster_acc,
            'train_ari': train_ari,
            'val_ari': val_ari,
            'train_silhouette': train_sil_score,
            'val_silhouette': val_sil_score,
            'trained_epochs': trained_epochs,
        })

    # Aggregate results across folds for the current parameter set
    df_fold_results = pd.DataFrame(fold_results)
    
    result_entry = params.copy()
    # Calculate mean and std for each metric
    for metric in df_fold_results.columns:
        result_entry[f'mean_{metric}'] = df_fold_results[metric].mean()
        result_entry[f'std_{metric}'] = df_fold_results[metric].std()

    hyperparam_results.append(result_entry)
    
    # Print the average results for the current hyperparameter combination
    tqdm.write("-" * 80)
    tqdm.write(f"Params: {params}")
    tqdm.write(f"--> Avg Val Accuracy: {result_entry['mean_val_acc']:.4f} ± {result_entry['std_val_acc']:.4f} | "
               f"Avg Val ARI: {result_entry['mean_val_ari']:.4f} ± {result_entry['std_val_ari']:.4f}")


print("\nHyperparameter tuning complete.")

Starting Hyperparameter Tuning with 5-Fold Cross-Validation...


Hyperparameter Combinations:   0%|          | 0/27 [00:00<?, ?it/s]

--------------------------------------------------------------------------------
Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.0001}
--> Avg Val Accuracy: 0.3306 ± 0.0210 | Avg Val ARI: 0.0008 ± 0.0050
--------------------------------------------------------------------------------
Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.001}
--> Avg Val Accuracy: 0.3590 ± 0.0095 | Avg Val ARI: 0.0071 ± 0.0054
--------------------------------------------------------------------------------
Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.01}
--> Avg Val Accuracy: 0.3225 ± 0.0184 | Avg Val ARI: 0.0076 ± 0.0268
--------------------------------------------------------------------------------
Params: {'decoder_dim': [], 'dropout': 0.2, 'weight_decay': 0.0001}
--> Avg Val Accuracy: 0.3469 ± 0.0356 | Avg Val ARI: 0.0038 ± 0.0150
--------------------------------------------------------------------------------
Params: {'decoder_dim': [], 'dropout': 0.2, 'weigh

### 2.5. Analyze Results

After completing the grid search, we analyze the results to find the best performing hyperparameter combination. We sort the results by the mean validation accuracy and display them in a summary table.

In [5]:
# --- Create and Display Summary DataFrame ---
summary_df = pd.DataFrame(hyperparam_results)
# The 'decoder_dim' column contains lists, which can be verbose. Convert to string for better display.
summary_df['decoder_dim'] = summary_df['decoder_dim'].astype(str)
summary_df = summary_df.sort_values(by='mean_val_acc', ascending=False)

print("\n" + "="*80)
print("===== Hyperparameter Tuning Summary (Top 5) =====")
display(summary_df.head())


# --- Find and Print Best Hyperparameters ---
best_result = summary_df.iloc[0]
print("\n" + "="*80)
print("===== Best Hyperparameters Found (based on Mean Validation Accuracy) =====")

# Drop metric columns to cleanly show the best parameter set
best_params_dict = best_result.drop([
    col for col in best_result.index if 'mean_' in col or 'std_' in col
]).to_dict()

print(f"Best Parameters: {best_params_dict}")
print(f"Best {N_SPLITS}-Fold CV Mean Accuracy: {best_result['mean_val_acc']:.4f} ± {best_result['std_val_acc']:.4f}")
print("="*80 + "\n")


# --- Save full results to CSV ---
results_path = config.OUTPUT_DIR / "hyperparameter_tuning_results.csv"
summary_df.to_csv(results_path, index=False)
print(f"Full hyperparameter tuning results saved to:\n{results_path}")



===== Hyperparameter Tuning Summary (Top 5) =====


,decoder_dim,dropout,weight_decay,mean_train_loss,std_train_loss,mean_val_loss,std_val_loss,mean_train_acc,std_train_acc,mean_val_acc,...,mean_train_ari,std_train_ari,mean_val_ari,std_val_ari,mean_train_silhouette,std_train_silhouette,mean_val_silhouette,std_val_silhouette,mean_trained_epochs,std_trained_epochs
1,[],0.1,0.0010,0.032660,0.012944,1.181603,0.133382,0.650445,0.165837,0.359039,...,0.395273,0.221956,0.007132,0.005373,0.135802,0.009132,0.095773,0.015814,35.0,26.842131
19,[64],0.1,0.0010,0.041121,0.007108,1.190253,0.123660,0.621633,0.114030,0.350979,...,0.356151,0.125112,0.003317,0.013785,0.137982,0.007175,0.101569,0.017216,23.6,5.272571
15,[32],0.3,0.0001,0.165799,0.031206,1.139503,0.125413,0.561869,0.078287,0.348918,...,0.295345,0.059892,0.008086,0.027200,0.174074,0.005833,0.137093,0.009563,24.4,4.505552
3,[],0.2,0.0001,0.071013,0.006750,1.201288,0.154893,0.709386,0.129836,0.346918,...,0.444850,0.175782,0.003762,0.015041,0.150426,0.003358,0.134841,0.012087,23.8,2.949576
25,[64],0.3,0.0010,0.164262,0.019808,1.132395,0.106112,0.594722,0.147392,0.346815,...,0.337802,0.172677,0.007409,0.019334,0.183573,0.008960,0.168869,0.020043,22.0,1.000000



===== Best Hyperparameters Found (based on Mean Validation Accuracy) =====
Best Parameters: {'decoder_dim': '[]', 'dropout': 0.1, 'weight_decay': 0.001}
Best 5-Fold CV Mean Accuracy: 0.3590 ± 0.0095

Full hyperparameter tuning results saved to:
/data02/jaejoon/T2D_subtype_analysis/outputs/hyperparameter_tuning_results.csv
